# PREPROCESSING DATA

In [155]:
# %load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

In [156]:
import numpy as np
import pandas as pd
from preprocessing import impute, get_impute_stats, clip, feature_engieneering, remove_first_column, save_processed

In [157]:
train_df = pd.read_csv('../data/raw/cs-training.csv', delimiter=',')
test_df = pd.read_csv('../data/raw/cs-test.csv', delimiter=',')


## Dropping age = 0

In [158]:
train_df = train_df[train_df['age'] >= 18]


## Using median for null values

MonthlyIncome & NumberOfDependents null values replaced with median from age bins 0-30, 30-45, 45-60, 60-75, 75-120

Clipping RevolvingUtilizationOfUnsecuredLines columns into (0, 1) - it is impossible to be out of this range, if it would have chance of having over it then i would leave it but it is mathematically impossible

In [159]:
train_df = impute(train_df, "MonthlyIncome")
train_df = impute(train_df, "NumberOfDependents")
train_df = clip(train_df, bounds=(0,1))

3321 values clipped in 'RevolvingUtilizationOfUnsecuredLines'


### making sure we dont get data leakage - using median from train_df:

In [160]:
test_df = impute(test_df, "MonthlyIncome", get_impute_stats(train_df, "MonthlyIncome"))
test_df = impute(test_df, "NumberOfDependents", get_impute_stats(train_df, "NumberOfDependents"))
test_df = clip(test_df, bounds=(0,1))

2181 values clipped in 'RevolvingUtilizationOfUnsecuredLines'


## Lil feature engieneering

MonthlyDebt = DebtRatio * MonthlyIncome

TotalLatePayments = NumberOfTime30-59DaysPastDueNotWorse + NumberOfTime60-89DaysPastDueNotWorse + NumberOfTimes90DaysLate

In [161]:
train_df = feature_engieneering(train_df)
test_df = feature_engieneering(test_df)

## Removing first column - indexes

In [162]:
train_df = remove_first_column(train_df)
test_df = remove_first_column(test_df)

# Saving processed data into csv

In [163]:
save_processed(train_df, '../data/processed/train.csv')
save_processed(test_df, '../data/processed/test.csv')

Saved 149999 rows to ../data/processed/train.csv
Saved 101503 rows to ../data/processed/test.csv
